In [1]:
#Started 14/8/2025 - Pokemon Worlds 2025 In Anaheim Started 1 Day later
import sys
from poke_env import RandomPlayer
from poke_env.data import GenData
from poke_env import AccountConfiguration
from poke_env.environment import DoublesEnv
from poke_env.battle import AbstractBattle, Battle


import numpy as np
import numpy.typing as npt
import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings("ignore")

from collections import defaultdict
from typing import Any, Dict, Optional

from gymnasium.spaces import Box, Discrete, Space
import torch
import torch.nn as nn
from torch import multiprocessing
from tensordict import TensorDict, TensorDictBase
from tensordict.nn import TensorDictModule
from tensordict.nn.distributions import NormalParamExtractor

from ray.rllib.algorithms import PPOConfig
from ray.rllib.core import Columns
from ray.rllib.core.rl_module import RLModuleSpec
from ray.rllib.core.rl_module.apis.value_function_api import ValueFunctionAPI
from ray.rllib.core.rl_module.torch import TorchRLModule
from ray.rllib.env import ParallelPettingZooEnv
from ray.tune.registry import register_env

from torchrl.collectors import SyncDataCollector
from torchrl.data.replay_buffers import ReplayBuffer
from torchrl.data.replay_buffers.samplers import SamplerWithoutReplacement
from torchrl.data.replay_buffers.storages import LazyTensorStorage
from torchrl.envs import (Compose, DoubleToFloat, ObservationNorm, StepCounter,
                          TransformedEnv)


#Custom Env pytorch tutorial
from typing import Optional
from torchrl.data import BoundedTensorSpec, CompositeSpec, UnboundedContinuousTensorSpec
from torchrl.envs import (
    CatTensors,
    EnvBase,
    Transform,
    TransformedEnv,
    UnsqueezeTransform,
)
from torchrl.envs.transforms.transforms import _apply_to_composite
from torchrl.envs.utils import step_mdp
#---------------------------
from torchrl.envs.libs.gym import GymEnv
from torchrl.envs.utils import check_env_specs, ExplorationType, set_exploration_type
from torchrl.modules import ProbabilisticActor, TanhNormal, ValueOperator
from torchrl.objectives import ClipPPOLoss
from torchrl.objectives.value import GAE
from tqdm import tqdm

In [2]:
from torchrl.envs.libs.pettingzoo import PettingZooWrapper

In [23]:
is_fork = multiprocessing.get_start_method() == "fork"
device = (
    torch.device(0)
    if torch.cuda.is_available() and not is_fork
    else torch.device("cpu")
)
num_cells = 256  # number of cells in each layer i.e. output dim.
lr = 3e-4
max_grad_norm = 1.0

In [24]:
frames_per_batch = 1000
# For a complete training, bring the number of frames up to 1M
total_frames = 50_000

In [25]:
sub_batch_size = 64  # cardinality of the sub-samples gathered from the current data in the inner loop
num_epochs = 10  # optimization steps per batch of data collected
clip_epsilon = (
    0.2  # clip value for PPO loss: see the equation in the intro for more context.
)
gamma = 0.99
lmbda = 0.95
entropy_eps = 1e-4

In [26]:
#Custom Env pytorch tutorial
DEFAULT_HIGH = [3, 3, 3, 3, 4, 4, 4, 4, 1, 1]
DEFAULT_LOW = [-1, -1, -1, -1, 0, 0, 0, 0, 0, 0]

In [6]:
teams =[ """
Typhlosion-Hisui @ Charcoal  
Ability: Blaze  
Level: 50  
Tera Type: Fire  
EVs: 252 SpA / 4 SpD / 252 Spe  
Timid Nature  
IVs: 0 Atk  
- Eruption  
- Heat Wave  
- Shadow Ball  
- Protect  

Whimsicott @ Covert Cloak  
Ability: Prankster  
Level: 50  
Tera Type: Dark  
EVs: 244 HP / 4 Def / 4 SpA / 4 SpD / 252 Spe  
Timid Nature  
IVs: 0 Atk  
- Tailwind  
- Moonblast  
- Sunny Day  
- Encore  

Ursaluna-Bloodmoon @ Life Orb  
Ability: Mind's Eye  
Level: 50  
Tera Type: Normal  
EVs: 4 HP / 252 SpA / 252 Spe  
Timid Nature  
IVs: 0 Atk  
- Hyper Voice  
- Blood Moon  
- Earth Power  
- Protect  

Primarina @ Throat Spray  
Ability: Liquid Voice  
Level: 50  
Tera Type: Grass  
EVs: 244 HP / 52 Def / 108 SpA / 28 SpD / 76 Spe  
Modest Nature  
IVs: 0 Atk  
- Moonblast  
- Hyper Voice  
- Haze  
- Protect  

Meowscarada @ Focus Sash  
Ability: Protean  
Level: 50  
Tera Type: Grass  
EVs: 4 HP / 252 Atk / 252 Spe  
Jolly Nature  
- Flower Trick  
- Knock Off  
- U-turn  
- Protect  

Farigiraf @ Safety Goggles  
Ability: Armor Tail  
Level: 50  
Tera Type: Fire  
EVs: 228 HP / 156 Def / 124 SpD  
Relaxed Nature  
IVs: 0 Atk / 0 Spe  
- Psychic Noise  
- Hyper Voice  
- Helping Hand  
- Trick Room  
""",

"""
Annihilape @ Lum Berry  
Ability: Defiant  
Level: 50  
Tera Type: Water  
EVs: 180 HP / 36 Atk / 12 Def / 28 SpD / 252 Spe  
Adamant Nature  
- Rage Fist  
- Drain Punch  
- Bulk Up  
- Protect  

Maushold @ Safety Goggles  
Ability: Friend Guard  
Level: 50  
Tera Type: Ghost  
EVs: 252 HP / 4 Def / 252 Spe  
Timid Nature  
- Follow Me  
- Beat Up  
- Taunt  
- Protect  

Sinistcha @ Sitrus Berry  
Ability: Hospitality  
Level: 50  
Tera Type: Fairy  
EVs: 236 HP / 36 Def / 236 SpD  
Sassy Nature  
IVs: 0 Atk / 0 Spe  
- Matcha Gotcha  
- Life Dew  
- Trick Room  
- Rage Powder  

Archaludon @ Assault Vest  
Ability: Stamina  
Level: 50  
Tera Type: Grass  
EVs: 212 HP / 12 Def / 44 SpA / 212 SpD / 28 Spe  
Modest Nature  
- Electro Shot  
- Flash Cannon  
- Body Press  
- Draco Meteor  

Pelipper @ Focus Sash  
Ability: Drizzle  
Level: 50  
Tera Type: Stellar  
EVs: 252 SpA / 4 SpD / 252 Spe  
Modest Nature  
- Hurricane  
- Weather Ball  
- Wide Guard  
- Protect  

Hydreigon @ Scope Lens  
Ability: Levitate  
Level: 50  
Shiny: Yes  
Tera Type: Steel  
EVs: 252 SpA / 4 SpD / 252 Spe  
Timid Nature  
- Draco Meteor  
- Dark Pulse  
- Focus Energy  
- Protect  

"""
]

In [ ]:
player_1_config = AccountConfiguration("Player_1", None)
random_player1 = RandomPlayer(account_configuration=player_1_config, max_concurrent_battles=1, team=teams[0], battle_format="gen9vgc2025regh", log_level=20)

In [ ]:
player_2_config = AccountConfiguration("Player_2", None)
random_player2 = RandomPlayer(account_configuration=player_2_config, max_concurrent_battles=1, team=teams[1], battle_format="gen9vgc2025regh", log_level=20)

In [ ]:
await random_player1.battle_against(random_player2, n_battles=1)

Setup Basic VGC Env

In [4]:
vgc_format="gen9vgc2025regh"

In [ ]:
def _make_spec(self, td_params):
    # Under the hood, thi*s will populate self.output_spec["observation"]
    self.observation_spec = CompositeSpec(
        th=BoundedTensorSpec(
            low=-torch.pi,
            high=torch.pi,
            shape=(),
            dtype=torch.float32,
        ),
        thdot=BoundedTensorSpec(
            low=-td_params["params", "max_speed"],
            high=td_params["params", "max_speed"],
            shape=(),
            dtype=torch.float32,
        ),
        # we need to add the ``params`` to the observation specs, as we want
        # to pass it at each step during a rollout
        params=make_composite_from_td(td_params["params"]),
        shape=(),
    )
    # since the environment is stateless, we expect the previous output as input.
    # For this, ``EnvBase`` expects some state_spec to be available
    self.state_spec = self.observation_spec.clone()
    # action-spec will be automatically wrapped in input_spec when
    # `self.action_spec = spec` will be called supported
    self.action_spec = BoundedTensorSpec(
        low=-td_params["params", "max_torque"],
        high=td_params["params", "max_torque"],
        shape=(1,),
        dtype=torch.float32,
    )
    self.reward_spec = UnboundedContinuousTensorSpec(shape=(*td_params.shape, 1))


def make_composite_from_td(td):
    # custom function to convert a ``tensordict`` in a similar spec structure
    # of unbounded values.
    composite = CompositeSpec(
        {
            key: make_composite_from_td(tensor)
            if isinstance(tensor, TensorDictBase)
            else UnboundedContinuousTensorSpec(
                dtype=tensor.dtype, device=tensor.device, shape=tensor.shape
            )
            for key, tensor in td.items()
        },
        shape=td.shape,
    )
    return composite

In [32]:
def _set_seed(self, seed: Optional[int]):
    rng = torch.manual_seed(seed)
    self.rng = rng

In [ ]:
def _step(tensordict):
    obs, re, term, trunc, add_info = tensordict["PokeEnv"].step(tensordict["actions"])
    out = TensorDict({
        "observation": obs,
        "reward":re,
        "done":term,
        "truncated":trunc,
        "additional_info":add_info,
        },
        tensordict.shape,
    )
    return out

In [59]:
def _reset(self, tensordict):
    obs, add_info = tensordict["PokeEnv"].reset(self.seed)
    out = TensorDict({
        "observation": obs,
        "additional_info":add_info,
        },
        tensordict.shape,
    )
    return out

In [69]:
def gen_env(batch_size = None) -> TensorDictBase:
    if batch_size is None:
        batch_size = []

    td = TensorDict(
        {
            "PokeEnv" : DoublesEnv(battle_format=vgc_format),
        },
        [],
    )
    if batch_size:
        td = td.expand(batch_size).contiguous()
    return td

In [70]:
gen_env()

TensorDict(
    fields={
        PokeEnv: NonTensorData(data=DoublesEnv, batch_size=torch.Size([]), device=None)},
    batch_size=torch.Size([]),
    device=None,
    is_shared=False)

In [ ]:
class PendulumEnv(EnvBase):

    def __init__(self, env=None, seed=None, device="cpu"):
        if env is None:
            env = self.gen_env()

        super().__init__(device=device, batch_size=[])
        if seed is None:
            seed = torch.empty((), dtype=torch.int64).random_().item()
        self.set_seed(seed)

    # Helpers: _make_step and gen_params
    #_make_spec = _make_spec
    gen_env=gen_env

    # Mandatory methods: _step, _reset and _set_seed
    _reset = _reset
    _step = _step
    _set_seed = _set_seed

In [68]:
env = PendulumEnv()

In [56]:
env.agent1

In [51]:
test_dict = TensorDict({
    "PokeEnv":env
})

In [57]:
test_dict["PokeEnv"].agent1

In [21]:
class VGC_Env(DoublesEnv[npt.NDArray[np.float32]]):
    LOW = [-1, -1, -1, -1, 0, 0, 0, 0, 0, 0]
    HIGH = [3, 3, 3, 3, 4, 4, 4, 4, 1, 1]

    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.observation_spaces = {
            agent: Box(
                np.array(self.LOW, dtype=np.float32),
                np.array(self.HIGH, dtype=np.float32),
                dtype=np.float32,
            )
            for agent in self.possible_agents
        }
    
    @classmethod
    def create_multi_agent_env(cls, config: Dict[str, Any]) -> ParallelPettingZooEnv:
        env = cls(
            battle_format=config["battle_format"],
            log_level=25,
            open_timeout=None,
            strict=False,
        )
        return ParallelPettingZooEnv(env)

    def calc_reward(self, battle) -> float:
        return self.reward_computing_helper(
            battle, fainted_value=2.0, hp_value=1.0, victory_value=30.0
        )

    def embed_battle(self, battle: AbstractBattle):
        assert isinstance(battle, Battle)
        # -1 indicates that the move does not have a base power
        # or is not available
        moves_base_power = -np.ones(4)
        moves_dmg_multiplier = np.ones(4)
        for i, move in enumerate(battle.available_moves):
            moves_base_power[i] = (
                move.base_power / 100
            )  # Simple rescaling to facilitate learning
            if battle.opponent_active_pokemon is not None:
                moves_dmg_multiplier[i] = move.type.damage_multiplier(
                    battle.opponent_active_pokemon.type_1,
                    battle.opponent_active_pokemon.type_2,
                    type_chart=battle.opponent_active_pokemon._data.type_chart,
                )

        # We count how many pokemons have fainted in each team
        fainted_mon_team = len([mon for mon in battle.team.values() if mon.fainted]) / 6
        fainted_mon_opponent = (
            len([mon for mon in battle.opponent_team.values() if mon.fainted]) / 6
        )

        # Final vector with 10 components
        final_vector = np.concatenate(
            [
                moves_base_power,
                moves_dmg_multiplier,
                [fainted_mon_team, fainted_mon_opponent],
            ]
        )
        return np.float32(final_vector)

In [19]:
env = DoublesEnv(battle_format=vgc_format, save_replays=True, accept_open_team_sheet=True, start_timer_on_battle_start=True)

In [29]:
PettingZooWrapper(env=env, use_mask=True)

ImportError: could not set anything related to gym backend gymnasium with version=1.0.0 for the function with name torchrl.envs.common.EnvBase._register_gym. Check that the gym versions match!

In [27]:
env = TransformedEnv(
    env, 
    Compose(
        ObservationNorm(in_keys=["observation"]),
        DoubleToFloat(),
        StepCounter(),
    ),
)

AttributeError: 'DoublesEnv' object has no attribute 'device'